In [1]:
import json
import requests
import random
import string
import secrets
import time
import re
import collections
import os
from typing import List, Dict, Set, Optional

try:
    from urllib.parse import parse_qs, urlencode, urlparse
except ImportError:
    from urlparse import parse_qs, urlparse
    from urllib import urlencode

from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# ---- Torch imports for the MoE hangman solver ----
import torch
import torch.nn as nn


class HangmanAPI(object):
    """
    Hangman API client with a gated Mixture-of-Experts (MoE) inference policy.

    Loads:
      - expert_short_bilstm.pt
      - expert_medium_bilstm.pt
      - expert_long_bilstm.pt
      - expert_common_bilstm.pt
      - expert_rare_bilstm.pt
      - best_gating_network.pt

    The inference logic mirrors the training/eval code you provided:
      - Each expert outputs per-position logits over 26 letters.
      - We softmax, then average probabilities over blank positions only.
      - Gate consumes [state_features(29) + concat(expert_probs)(130)] => 159 dims
      - Gate softmax gives 5 weights; we form weighted sum of expert probs.
      - Mask already-guessed letters; pick argmax.
    """

    # -------------------- Vocabulary (MUST match training) --------------------
    _CHARS = string.ascii_lowercase
    _CHAR_TO_IDX = {ch: i + 1 for i, ch in enumerate(_CHARS)}  # a..z => 1..26
    _CHAR_TO_IDX["_"] = 27
    _CHAR_TO_IDX["<PAD>"] = 0
    _IDX_TO_CHAR = {idx: ch for ch, idx in _CHAR_TO_IDX.items()}
    _VOCAB_SIZE = len(_CHAR_TO_IDX)  # 28

    # -------------------- Model hyperparams (MUST match training) --------------------
    _EMBEDDING_DIM = 128
    _HIDDEN_DIM = 512
    _NUM_LAYERS = 2

    _EXPERT_NAMES = ["short", "medium", "long", "common", "rare"]

    class HangmanBiLSTM(nn.Module):
        def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int, num_layers: int = 2, padding_idx: int = 0):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
            self.lstm = nn.LSTM(
                embedding_dim,
                hidden_dim,
                num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=0.2
            )
            self.fc = nn.Linear(hidden_dim * 2, 26)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            emb = self.embedding(x)
            out, _ = self.lstm(emb)
            return self.fc(out)

    class GatingNetwork(nn.Module):
        def __init__(self, input_dim: int = 159, hidden_dims: List[int] = [256, 128], output_dim: int = 5):
            super().__init__()
            layers = []
            prev = input_dim
            for h in hidden_dims:
                layers.append(nn.Linear(prev, h))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(0.2))
                prev = h
            layers.append(nn.Linear(prev, output_dim))
            self.net = nn.Sequential(*layers)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.net(x)  # raw logits

    def __init__(self, access_token=None, session=None, timeout=None):
        # --- Original fields ---
        self.hangman_url = self.determine_hangman_url()
        self.access_token = access_token
        self.session = session or requests.Session()
        self.timeout = timeout
        self.guessed_letters = []

        full_dictionary_location = "words_250000_train.txt"
        self.full_dictionary = self.build_dictionary(full_dictionary_location)
        self.full_dictionary_common_letter_sorted = collections.Counter("".join(self.full_dictionary)).most_common()
        self.current_dictionary = []

        # --- MoE inference fields ---
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Resolve model directory relative to this file (safer in most runners)
        if "__file__" in globals():
            self._base_dir = os.path.dirname(os.path.abspath(__file__))
        else:
            self._base_dir = os.getcwd()

        self.expert_models: List[nn.Module] = []
        self.gate: Optional[nn.Module] = None
        self._models_loaded: bool = False

        self._load_models()

    @staticmethod
    def determine_hangman_url():
        links = ['https://trexsim.com']
        data = {link: 0 for link in links}

        for link in links:
            requests.get(link)
            for i in range(10):
                s = time.time()
                requests.get(link)
                data[link] = time.time() - s

        link = sorted(data.items(), key=lambda x: x[1])[0][0]
        link += '/trexsim/hangman'
        return link

    # -------------------- MoE helpers --------------------
    def _load_models(self) -> None:
        """
        Load 5 experts + gate. If loading fails, we keep _models_loaded=False
        and guess() will fall back to the baseline dictionary-frequency approach.
        """
        try:
            experts: Dict[str, nn.Module] = {}
            for name in self._EXPERT_NAMES:
                model = self.HangmanBiLSTM(
                    self._VOCAB_SIZE,
                    self._EMBEDDING_DIM,
                    self._HIDDEN_DIM,
                    num_layers=self._NUM_LAYERS,
                    padding_idx=self._CHAR_TO_IDX["<PAD>"],
                ).to(self.device)

                path = os.path.join(self._base_dir, f"expert_{name}_bilstm.pt")
                if not os.path.exists(path):
                    raise FileNotFoundError(f"Expert file not found: {path}")

                state = torch.load(path, map_location=self.device)
                model.load_state_dict(state)
                model.eval()
                experts[name] = model

            # Maintain the expert ordering used in training code
            self.expert_models = [experts[n] for n in self._EXPERT_NAMES]

            gate_path = os.path.join(self._base_dir, "best_gating_network.pt")
            if not os.path.exists(gate_path):
                raise FileNotFoundError(f"Gating model file not found: {gate_path}")

            gate = self.GatingNetwork(input_dim=159, hidden_dims=[256, 128], output_dim=5).to(self.device)
            gate_state = torch.load(gate_path, map_location=self.device)
            gate.load_state_dict(gate_state)
            gate.eval()
            self.gate = gate

            self._models_loaded = True
        except Exception as e:
            # Keep the API usable even if models aren't present
            self._models_loaded = False
            self.expert_models = []
            self.gate = None
            # Print once; many evaluation harnesses ignore stdout anyway
            print(f"[WARN] MoE models not loaded, falling back to baseline guess(). Reason: {e}")

    def _parse_masked_word(self, word: str) -> List[str]:
        """
        Server passes patterns like: "_ p p _ e " (space separated).
        We strip whitespace and return list of characters like ['_', 'p', 'p', '_', 'e'].
        """
        clean = re.sub(r"\s+", "", word.strip())
        return list(clean)

    def _get_expert_probs(self, input_tensor: torch.Tensor) -> List[torch.Tensor]:
        """
        For input_tensor shape (1, L), returns list of 5 tensors each shape (26,),
        equal to expert's average probability over masked ('_') positions.
        """
        # mask: (1, L)
        mask = (input_tensor == self._CHAR_TO_IDX["_"]).float()
        probs_list: List[torch.Tensor] = []

        with torch.no_grad():
            for model in self.expert_models:
                logits = model(input_tensor)               # (1, L, 26)
                probs = torch.softmax(logits, dim=-1)      # (1, L, 26)
                masked_probs = (probs * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                probs_list.append(masked_probs.squeeze(0))  # (26,)

        return probs_list

    def _get_state_features(self, input_seq: List[int], guessed_letters: Set[str], incorrect_count: int) -> torch.Tensor:
        """
        Build the 29-dim state vector exactly like training:
          [length_norm, blanks_norm, incorrect_norm] + guessed_vec(26)
        """
        L = len(input_seq)
        blanks = sum(1 for idx in input_seq if idx == self._CHAR_TO_IDX["_"])

        length_norm = L / 20.0
        blanks_norm = (blanks / L) if L > 0 else 0.0
        incorrect_norm = incorrect_count / 6.0

        guessed_vec = torch.zeros(26, device=self.device)
        for ch in guessed_letters:
            if ch in self._CHAR_TO_IDX and ch != "_" and ch != "<PAD>":
                guessed_vec[self._CHAR_TO_IDX[ch] - 1] = 1.0

        base = torch.tensor([length_norm, blanks_norm, incorrect_norm], dtype=torch.float32, device=self.device)
        return torch.cat([base, guessed_vec])  # (29,)

    # -------------------- Guess policy --------------------
    def guess(self, word):  # word input example: "_ p p _ e "
        """
        MoE gated inference guess.
        If models aren't available, falls back to the original baseline approach.
        """
        # If MoE isn't loaded, use baseline dictionary approach
        if not self._models_loaded:
            return self._baseline_guess(word)

        board = self._parse_masked_word(word)  # list of chars including '_'
        revealed_letters = set([c for c in board if c != "_"])

        guessed_set = set(self.guessed_letters)

        # Incorrect guesses = guessed letters that do not appear in revealed pattern
        incorrect_count = sum(1 for ch in guessed_set if ch not in revealed_letters)

        # Convert board to indices
        input_seq = []
        for c in board:
            if c == "_":
                input_seq.append(self._CHAR_TO_IDX["_"])
            else:
                # defensive: lower it; server words are lower-case
                c = c.lower()
                if c in self._CHAR_TO_IDX:
                    input_seq.append(self._CHAR_TO_IDX[c])
                else:
                    # unknown char shouldn't happen; treat as blank
                    input_seq.append(self._CHAR_TO_IDX["_"])

        input_tensor = torch.tensor([input_seq], dtype=torch.long, device=self.device)  # (1, L)

        # Expert probabilities averaged over blanks
        probs_list = self._get_expert_probs(input_tensor)  # 5 * (26,)

        # Gate features: 29 + 130 = 159
        base_feat = self._get_state_features(input_seq, guessed_set, incorrect_count)  # (29,)
        expert_feat = torch.cat(probs_list, dim=0)  # (130,)
        full_feat = torch.cat([base_feat, expert_feat], dim=0).unsqueeze(0)  # (1, 159)

        # Gate weights (softmax over logits)
        with torch.no_grad():
            gate_logits = self.gate(full_feat).squeeze(0)  # (5,)
            gate_weights = torch.softmax(gate_logits, dim=0)  # (5,)

        # Weighted sum of expert probs
        weighted_probs = torch.zeros(26, device=self.device)
        for w, p in zip(gate_weights, probs_list):
            weighted_probs += w * p

        # Mask already-guessed letters
        for ch in guessed_set:
            if ch in self._CHAR_TO_IDX and ch in string.ascii_lowercase:
                weighted_probs[self._CHAR_TO_IDX[ch] - 1] = -float("inf")

        # If everything is masked (shouldn't happen), pick any remaining letter
        if torch.isinf(weighted_probs).all():
            for ch in string.ascii_lowercase:
                if ch not in guessed_set:
                    return ch
            return "e"  # absolute last resort

        guess_idx = torch.argmax(weighted_probs).item()  # 0..25
        guess_char = self._IDX_TO_CHAR[guess_idx + 1]   # 1..26 -> 'a'..'z'
        return guess_char

    # -------------------- Baseline fallback (original logic, preserved) --------------------
    def _baseline_guess(self, word: str) -> str:
        # clean the word so that we strip away the space characters
        # replace "_" with "." as "." indicates any character in regular expressions
        clean_word = word[::2].replace("_", ".")

        # find length of passed word
        len_word = len(clean_word)

        # grab current dictionary of possible words from self object, initialize new possible words dictionary to empty
        current_dictionary = self.current_dictionary
        new_dictionary = []

        # iterate through all of the words in the old plausible dictionary
        for dict_word in current_dictionary:
            # continue if the word is not of the appropriate length
            if len(dict_word) != len_word:
                continue

            # if dictionary word is a possible match then add it to the current dictionary
            if re.match(clean_word, dict_word):
                new_dictionary.append(dict_word)

        # overwrite old possible words dictionary with updated version
        self.current_dictionary = new_dictionary

        # count occurrence of all characters in possible word matches
        full_dict_string = "".join(new_dictionary)
        c = collections.Counter(full_dict_string)
        sorted_letter_count = c.most_common()

        guess_letter = "!"

        # return most frequently occurring letter in all possible words that hasn't been guessed yet
        for letter, instance_count in sorted_letter_count:
            if letter not in self.guessed_letters:
                guess_letter = letter
                break

        # if no word matches in training dictionary, default back to ordering of full dictionary
        if guess_letter == "!":
            sorted_letter_count = self.full_dictionary_common_letter_sorted
            for letter, instance_count in sorted_letter_count:
                if letter not in self.guessed_letters:
                    guess_letter = letter
                    break

        return guess_letter

    ##########################################################
    # You'll likely not need to modify any of the code below #
    ##########################################################

    def build_dictionary(self, dictionary_file_location):
        text_file = open(dictionary_file_location, "r")
        full_dictionary = text_file.read().splitlines()
        text_file.close()
        return full_dictionary

    def start_game(self, practice=True, verbose=True):
        # reset guessed letters to empty set and current plausible dictionary to the full dictionary
        self.guessed_letters = []
        self.current_dictionary = self.full_dictionary

        response = self.request("/new_game", {"practice": practice})
        if response.get('status') == "approved":
            game_id = response.get('game_id')
            word = response.get('word')
            tries_remains = response.get('tries_remains')
            if verbose:
                print("Successfully start a new game! Game ID: {0}. # of tries remaining: {1}. Word: {2}.".format(
                    game_id, tries_remains, word))
            while tries_remains > 0:
                # get guessed letter from user code
                guess_letter = self.guess(word)

                # append guessed letter to guessed letters field in hangman object
                self.guessed_letters.append(guess_letter)
                if verbose:
                    print("Guessing letter: {0}".format(guess_letter))

                try:
                    res = self.request("/guess_letter", {"request": "guess_letter", "game_id": game_id, "letter": guess_letter})
                except HangmanAPIError:
                    print('HangmanAPIError exception caught on request.')
                    continue
                except Exception as e:
                    print('Other exception caught on request.')
                    raise e

                if verbose:
                    print("Sever response: {0}".format(res))
                status = res.get('status')
                tries_remains = res.get('tries_remains')
                if status == "success":
                    if verbose:
                        print("Successfully finished game: {0}".format(game_id))
                    return True
                elif status == "failed":
                    reason = res.get('reason', '# of tries exceeded!')
                    if verbose:
                        print("Failed game: {0}. Because of: {1}".format(game_id, reason))
                    return False
                elif status == "ongoing":
                    word = res.get('word')
        else:
            if verbose:
                print("Failed to start a new game")
        return status == "success"

    def my_status(self):
        return self.request("/my_status", {})

    def request(self, path, args=None, post_args=None, method=None):
        if args is None:
            args = dict()
        if post_args is not None:
            method = "POST"

        # Add `access_token` to post_args or args if it has not already been included.
        if self.access_token:
            if post_args and "access_token" not in post_args:
                post_args["access_token"] = self.access_token
            elif "access_token" not in args:
                args["access_token"] = self.access_token

        time.sleep(0.2)

        num_retry, time_sleep = 50, 2
        for it in range(num_retry):
            try:
                response = self.session.request(
                    method or "GET",
                    self.hangman_url + path,
                    timeout=self.timeout,
                    params=args,
                    data=post_args,
                    verify=False
                )
                break
            except requests.HTTPError as e:
                response = json.loads(e.read())
                raise HangmanAPIError(response)
            except requests.exceptions.SSLError as e:
                if it + 1 == num_retry:
                    raise
                time.sleep(time_sleep)

        headers = response.headers
        if 'json' in headers['content-type']:
            result = response.json()
        elif "access_token" in parse_qs(response.text):
            query_str = parse_qs(response.text)
            if "access_token" in query_str:
                result = {"access_token": query_str["access_token"][0]}
                if "expires" in query_str:
                    result["expires"] = query_str["expires"][0]
            else:
                raise HangmanAPIError(response.json())
        else:
            raise HangmanAPIError('Maintype was not text, or querystring')

        if result and isinstance(result, dict) and result.get("error"):
            raise HangmanAPIError(result)
        return result


class HangmanAPIError(Exception):
    def __init__(self, result):
        self.result = result
        self.code = None
        try:
            self.type = result["error_code"]
        except (KeyError, TypeError):
            self.type = ""

        try:
            self.message = result["error_description"]
        except (KeyError, TypeError):
            try:
                self.message = result["error"]["message"]
                self.code = result["error"].get("code")
                if not self.type:
                    self.type = result["error"].get("type", "")
            except (KeyError, TypeError):
                try:
                    self.message = result["error_msg"]
                except (KeyError, TypeError):
                    self.message = result

        Exception.__init__(self, self.message)

In [2]:
api = HangmanAPI(access_token="TREXQUANT_API_TOKEN", timeout=2000)

In [3]:
for i in range(200):
    print('Playing ', i, ' th game')
    api.start_game(practice=1, verbose=True)

    # DO NOT REMOVE as otherwise the server may lock you out for too high frequency of requests
    time.sleep(0.5)

Playing  0  th game
Successfully start a new game! Game ID: c612a7dada5e. # of tries remaining: 6. Word: _ _ _ _ _ _ _ _ _ _ .
Guessing letter: e
Sever response: {'game_id': 'c612a7dada5e', 'status': 'ongoing', 'tries_remains': 6, 'word': '_ _ _ _ e _ _ _ e _ '}
Guessing letter: r
Sever response: {'game_id': 'c612a7dada5e', 'status': 'ongoing', 'tries_remains': 6, 'word': '_ _ _ _ e r _ _ e _ '}
Guessing letter: s
Sever response: {'game_id': 'c612a7dada5e', 'status': 'ongoing', 'tries_remains': 6, 'word': '_ _ _ _ e r _ s e _ '}
Guessing letter: i
Sever response: {'game_id': 'c612a7dada5e', 'status': 'ongoing', 'tries_remains': 5, 'word': '_ _ _ _ e r _ s e _ '}
Guessing letter: o
Sever response: {'game_id': 'c612a7dada5e', 'status': 'ongoing', 'tries_remains': 4, 'word': '_ _ _ _ e r _ s e _ '}
Guessing letter: d
Sever response: {'game_id': 'c612a7dada5e', 'status': 'ongoing', 'tries_remains': 4, 'word': '_ _ _ _ e r _ s e d '}
Guessing letter: a
Sever response: {'game_id': 'c612a7dad